In [28]:
import pandas as pd
import numpy as np
import json

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import plotly.express as px

import math
from collections import Counter, defaultdict
import re
from unidecode import unidecode

In [29]:
df = pd.read_csv("../../data/papers.csv")
total_papers = len(df)
df.columns

Index(['Title', 'Database', 'Year', 'Month', 'Journal or Conference',
       'Paper Type', 'Data Source', 'Data Country', 'Data Domain',
       'Data Language', 'Data Availability', 'Link to Data', 'Data Details',
       'Size', 'Number of Prompts If Applicable', 'Medical Application',
       'Task Type', 'Topic', 'Note', 'Bias Evaluation Metric',
       'Bias Definition', 'Conclusions', 'Race or Ethnic Bias', 'Gender Bias',
       'Language Bias', 'Age Bias', 'Other Bias', 'LGBTQ+ Bias',
       'Disability Bias', 'Geography or Cultural Bias', 'Evaluated LLMs',
       'Reference Standard', 'Patient Inclusion', 'Has Debiasing',
       'Debias Focus', 'Debiasing Method', 'Debias Details'],
      dtype='str')

In [30]:
language_groups_lst = df["Data Language"].tolist()
language_groups_freqs = defaultdict(int)

In [31]:
for group in language_groups_lst:
    if not isinstance(group, str):
        continue
    # print(group, type(group))
    languages = group.split(",")
    for language in languages:
        language_groups_freqs[language] += 1

language_groups_freqs = dict(sorted(language_groups_freqs.items(), key=lambda item: item[1], reverse=True))

language_groups_freqs

{'English': 71,
 'Chinese': 3,
 'Arabic': 3,
 ' Spanish': 3,
 ' English': 3,
 'French': 2,
 ' Japanese': 2,
 'Traditional Chinese': 1,
 ' Simplified Chinese': 1,
 'Polish': 1,
 'Engllish': 1,
 ' Russian': 1,
 ' Kazakh': 1,
 ' Dutch': 1,
 'Mandarin Chinese': 1}

In [32]:
# Normalize language names so whitespace/typos/Chinese variants merge into one key per language.
_LANG_ALIASES = {
    "engllish": "english",
    "traditional chinese": "chinese",
    "simplified chinese": "chinese",
    "mandarin chinese": "chinese",
}

_normalized = defaultdict(int)
for _lang, _count in language_groups_freqs.items():
    _key = _lang.strip().lower()
    _key = _LANG_ALIASES.get(_key, _key)
    _normalized[_key] += _count

language_groups_freqs = dict(sorted(_normalized.items(), key=lambda kv: kv[1], reverse=True))
language_groups_freqs

{'english': 75,
 'chinese': 6,
 'arabic': 3,
 'spanish': 3,
 'french': 2,
 'japanese': 2,
 'polish': 1,
 'russian': 1,
 'kazakh': 1,
 'dutch': 1}

In [33]:
# countries.csv is the single source of truth: one country, one primary language.
countries_df = pd.read_csv("../languages_to_countries/data/countries.csv")
iso_df = pd.read_csv("../languages_to_countries/data/iso.csv")[["alpha-2", "alpha-3"]]

countries_df["language_key"] = (
    countries_df["language"].fillna("").str.strip().str.lower()
)
countries_df = countries_df.merge(
    iso_df, left_on="iso2", right_on="alpha-2", how="left"
)
countries_df[["country", "iso2", "alpha-3", "language", "language_key"]].head()

,country,iso2,alpha-3,language,language_key
0,China,CN,CHN,Chinese,chinese
1,India,IN,IND,Hindi,hindi
2,United States,US,USA,English,english
3,Indonesia,ID,IDN,Indonesian,indonesian
4,Pakistan,PK,PAK,Urdu,urdu


In [34]:
# Each country gets the paper-language frequency for its single primary language.
# Countries whose language isn't represented in the paper dataset get 0.
country_to_freqs = {
    unidecode(row["country"]): int(language_groups_freqs.get(row["language_key"], 0))
    for _, row in countries_df.iterrows()
}
country_to_freqs

{'China': 6,
 'India': 0,
 'United States': 75,
 'Indonesia': 0,
 'Pakistan': 0,
 'Nigeria': 75,
 'Brazil': 0,
 'Bangladesh': 0,
 'Russia': 1,
 'Mexico': 3,
 'Japan': 2,
 'Ethiopia': 0,
 'Philippines': 0,
 'Democratic Republic of the Congo': 2,
 'Egypt': 3,
 'Vietnam': 0,
 'Iran': 0,
 'Germany': 0,
 'Turkey': 0,
 'Thailand': 0,
 'France': 2,
 'United Kingdom': 75,
 'Tanzania': 0,
 'Italy': 0,
 'South Africa': 0,
 'Myanmar': 0,
 'Kenya': 75,
 'South Korea': 0,
 'Colombia': 3,
 'Sudan': 3,
 'Uganda': 75,
 'Spain': 3,
 'Argentina': 3,
 'Algeria': 3,
 'Ukraine': 0,
 'Iraq': 3,
 'Afghanistan': 0,
 'Canada': 75,
 'Poland': 1,
 'Morocco': 3,
 'Angola': 0,
 'Saudi Arabia': 3,
 'Malaysia': 0,
 'Ghana': 75,
 'Mozambique': 0,
 'Peru': 3,
 'Yemen': 3,
 'Uzbekistan': 0,
 'Nepal': 0,
 'Venezuela': 3,
 'Cameroon': 75,
 'Ivory Coast': 2,
 'Madagascar': 0,
 'Australia': 75,
 'North Korea': 0,
 'Niger': 2,
 'Taiwan': 6,
 'Sri Lanka': 0,
 'Syria': 3,
 'Burkina Faso': 2,
 'Mali': 2,
 'Malawi': 75,
 'Zambi

In [35]:
map = countries_df.rename(columns={"alpha-3": "iso_alpha"})[
    ["country", "iso_alpha", "language_key"]
].copy()
map["country"] = map["country"].map(unidecode)
map["frequency"] = (
    map["language_key"].map(language_groups_freqs).fillna(0).astype(int)
)
map = map.drop(columns=["language_key"]).dropna(subset=["iso_alpha"]).reset_index(drop=True)
map

,country,iso_alpha,frequency
0,China,CHN,6
1,India,IND,0
2,United States,USA,75
3,Indonesia,IDN,0
4,Pakistan,PAK,0
...,...,...,...
247,Reunion,REU,2
248,South Georgia and the South Sandwich Islands,SGS,75
249,Heard Island and McDonald Islands,HMD,0
250,Bouvet Island,BVT,0


In [36]:
# Discrete yellow -> orange -> red ramp; gray = no papers.
FREQ_BIN_ORDER = ["0", "1–5", "6–10", "11–15", "16+"]

def _count_to_freq_bin(count):
    if count <= 0:
        return "0"
    if count <= 5:
        return "1–5"
    if count <= 10:
        return "6–10"
    if count <= 15:
        return "11–15"
    return "16+"

def _wrap_legend_label(bin_label, langs, max_line_len=52):
    if not langs:
        return bin_label
    lines = [f"{bin_label}: {langs[0]}"]
    for lang in langs[1:]:
        candidate = lines[-1] + ", " + lang
        if len(candidate) <= max_line_len:
            lines[-1] = candidate
        else:
            lines.append("      " + lang)
    return "<br>".join(lines)

s = map["frequency"].fillna(0).astype(float)

map["freq_bin"] = pd.Categorical(
    np.select(
        [s <= 0, s <= 5, s <= 10, s <= 15],
        ["0", "1–5", "6–10", "11–15"],
        default="16+",
    ),
    categories=FREQ_BIN_ORDER,
    ordered=True,
)

color_discrete_map = {
    "0": "#bdbdbd",
    "1–5": "#fff176",
    "6–10": "#ffca28",
    "11–15": "#fb8c00",
    "16+": "#bf360c",
}

# Group languages by paper-count bin for the legend.
_bin_languages = defaultdict(list)
for _lang, _count in language_groups_freqs.items():
    _bin_languages[_count_to_freq_bin(_count)].append(_lang.title())

_legend_labels = {}
for _bin in FREQ_BIN_ORDER:
    _langs = sorted(_bin_languages.get(_bin, []))
    _legend_labels[_bin] = _wrap_legend_label(_bin, _langs)

fig = px.choropleth(
    map,
    locations="iso_alpha",
    color="freq_bin",
    hover_name="country",
    category_orders={"freq_bin": FREQ_BIN_ORDER},
    color_discrete_map=color_discrete_map,
    labels={"freq_bin": "Paper count"},
)

for _trace in fig.data:
    if _trace.name in _legend_labels:
        _trace.name = _legend_labels[_trace.name]

fig.update_layout(
    width=1680,
    height=900,
    margin=dict(l=10, r=10, t=10, b=150),
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=-0.085,
        xanchor="center",
        x=0.5,
        title=dict(text="Paper count", font=dict(size=17)),
        font=dict(size=16),
        itemwidth=36,
    ),
)
fig.update_geos(domain=dict(x=[0, 1], y=[0, 1]))

fig.update_layout(title=None)
fig.write_image("data_languages.svg")
fig.show()

/var/folders/04/2hwl1j9x5yl1hdpnd9k3jlqw0000gn/T/ipykernel_89499/3533176074.py:89: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


